# LI-6800 control

Two ways to talk to the instrument:

- **Native MQTT (primary, recommended)** — `li_mqtt.LI6800` connects directly to the LI-6800's internal MQTT bus over link-local IPv6. Real-time data, direct setpoints, nothing to start on the console. This is the same bus the touchscreen itself uses.
- **SSH file-protocol (fallback)** — `li_connect.LiCor` drops JSON commands over SSH to the `RemoteEnvMeasure.py` background program. Works, but requires that BP to be started once on the console.

Run the install cell once, then use the MQTT section. **`li.set(...)` drives the chamber immediately** — treat it like turning a knob on the console.

In [1]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached invoke-3.0.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached pynacl-1.6.2-cp38-abi3-win_amd64.whl.metadata (10 kB)
Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl (150 kB)
Using cached invoke-3.0.3-py3-none-any.whl (160 kB)
Using cached pynacl-1.6.2-cp38-abi3-win_amd64.whl (239 kB)

   ---------------------------------------- 0/4 [invoke]
   ---------------------------------------- 0/4 [invoke]
   ---------------------------------------- 0/4 [invoke]
   ---------------------------------------- 0/4 [invoke]
   ---------------------------------------- 0/4 [invoke]
   ---------------------------------------- 0/4 [invoke]
   ---------------------------------------- 0/4 [invoke]
   ---------------------------------------- 0/4 [invoke]
   ---------------------------------------- 0/4 [invoke]
   ----------------------------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Native MQTT control

In [2]:
from li_mqtt import LI6800

# Discovers the head's MQTT broker on the wired link (validates live data flow)
# and subscribes to the measurement streams. No password, no background program.
li = LI6800.connect()
print("connected to broker", li.broker)

connected to broker fe80::21c:94ff:fe04:8d5c%14


In [4]:
# Live values: raw measured (CO2_r, H2O_s, Tleaf, ...) merged with computed
# gas exchange (A, gsw, Ci, E, ...). Real-time; reading changes nothing.
vals = li.read()
for k in ("CO2_r", "CO2_s", "H2O_r", "H2O_s", "Tleaf", "Tchamber",
          "RHcham", "VPDleaf", "Flow", "Fan_speed", "A", "gsw", "Ci", "E"):
    print(f"  {k:<10} {vals.get(k)}")
print(f"({len(vals)} values total — see li.read().keys())")

  CO2_r      418.67
  CO2_s      420.061
  H2O_r      16.2101
  H2O_s      15.8615
  Tleaf      25.1706
  Tchamber   25.1528
  RHcham     50.34231808660229
  VPDleaf    1.5967983836313273
  Flow       399.745
  Fan_speed  10012.5
  A          -0.5898715761784118
  gsw        -0.0012487995057139701
  Ci         -336.6291418286978
  E          -2.0059959712805014e-05
(76 values total — see li.read().keys())


## CO2Dot CO₂-response experiment

For each target **sample CO₂** (`co2_s`): set it on the LI-6800 (over MQTT), wait for it to stabilise, wait an extra minute, then acquire one combined measurement — the LI-6800 gas-exchange values plus the CO2Dot's `status`, `spec_flash` (dark / light / difference spectra) and `env` — and save it to a timestamped JSON file under `co2dot_data/`.

Prerequisites: the `li` MQTT handle from the **Native MQTT control** section above must be connected, and the CO2Dot must be plugged in over USB. Run the cells below in order: connect the CO2Dot → define `acquire()` → single point (350) **or** the full loop.

In [5]:
import json, time
from datetime import datetime
from pathlib import Path

def acquire(li, dot, *, co2_s_setpoint=None, note="", save=True, outdir="co2dot_data"):
    """Read the LI-6800 + CO2Dot and return one record dict (with a timestamp).

    Captures: all live LI-6800 values, and the CO2Dot's status (spectrometer +
    BME settings), spec_flash (dark / lit / difference spectra), and env
    (BME688 T/P/RH/Gas). If save=True, also writes this single record to its own
    timestamped JSON file in outdir (handy for one-off points). The experiment
    loop passes save=False and gathers all records into one file instead.
    """
    record = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "co2_s_setpoint": co2_s_setpoint,
        "note": note,
        "licor": li.read(),
        "co2dot": {
            "status": dot.status(),            # spectrometer model/gain/atime/astep/led + BME status
            "spec_flash": dot.spec_flash(1),   # dark / lit / diff spectra (LED current 1)
            "env": dot.env(),                  # BME688 T / P / RH / Gas
        },
    }
    if save:
        Path(outdir).mkdir(parents=True, exist_ok=True)
        tag = f"_co2s{int(co2_s_setpoint)}" if co2_s_setpoint is not None else ""
        path = Path(outdir) / (datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + tag + ".json")
        path.write_text(json.dumps(record, indent=2), encoding="utf-8")
        print("saved", path)
    return record

In [9]:
from co2dot import CO2Dot
import time 
# Autodetects the CO2Dot over USB serial (sends 'hello', matches the device greeting).
dot = CO2Dot.connect()
print("CO2Dot on", dot.port)
time.sleep(1)  # wait for the dot to be ready
print("status:", dot.status())

CO2Dot on COM8
status: {'spectrometer': {'model': 'AS7341', 'available': True, 'atime': 100, 'astep': 999, 'gain': 5, 'led': 0}, 'bme': {'available': True}}


In [11]:
# Single point: set co2_s = 350, wait to stabilise, wait an extra minute, then acquire.
SETPOINT = 380
li.set(co2_s=SETPOINT)
print(f"co2_s -> {SETPOINT} ppm; waiting to stabilise ...")
li.wait_stable("CO2_s", SETPOINT, tol=2, timeout_s=300)
print(f"stable (CO2_s={li.get('CO2_s')}); waiting an extra 60 s ...")
time.sleep(1)
record = acquire(li, dot, co2_s_setpoint=SETPOINT)

co2_s -> 380 ppm; waiting to stabilise ...
stable (CO2_s=380.027); waiting an extra 60 s ...
saved co2dot_data\2026-06-12_12-25-46_co2s380.json


In [ ]:
# Full experiment: iterate temperature, RH and co2_s setpoints.
# For each (T, RH, repeat, co2_s): set co2_s, then IMMEDIATELY record a time series of
# MEASUREMENTS points spaced DELAY_BETWEEN_MEASUREMENTS s apart (the CO2 step response).
# All points go into ONE timestamped JSON, rewritten after every point so partial data
# survives an interruption. Each record carries its own timestamp + temp/rh/rep/measurement.

TEMP_VALUES = [17, 23, 31]                 # °C chamber air temperature
RH_VALUES = [40, 50, 60, 70]               # %RH; edit as needed
CO2_S_VALUES = [300,330,360,390,420,450,480,510,480,450,420,390,360,330]   # ppm; edit as needed

MEASUREMENTS = 6                            # measurements recorded after each co2 step
DELAY_BETWEEN_MEASUREMENTS = 20             # s between measurements (6 x 20 s = 2 min)
REPEATS = 4
TEMP_TOL, TEMP_TIMEOUT = 0.5, 1200          # °C window / s max wait for temperature
RH_TOL, RH_TIMEOUT = 3, 1200                # %RH window / s max wait for humidity

TOTAL = len(TEMP_VALUES) * len(RH_VALUES) * REPEATS * len(CO2_S_VALUES) * MEASUREMENTS


outdir = Path("co2dot_data")
outdir.mkdir(parents=True, exist_ok=True)
started = datetime.now()
exp_path = outdir / (started.strftime("%Y-%m-%d_%H-%M-%S") + "_co2response.json")

experiment = {
    "experiment_start": started.isoformat(timespec="seconds"),
    "temp_values": TEMP_VALUES,
    "rh_values": RH_VALUES,
    "co2_s_values": CO2_S_VALUES,
    "repeats": REPEATS,
    "measurements": MEASUREMENTS,
    "delay_between_measurements_s": DELAY_BETWEEN_MEASUREMENTS,
    "records": [],
}


try:
    for temp in TEMP_VALUES:
        li.set(tair=temp)                       # was Tchamber (a read-only key -> KeyError)
        print(f"\nTair -> {temp} °C; waiting to stabilise ...")
        if not li.wait_stable("Tchamber", temp, tol=TEMP_TOL, timeout_s=TEMP_TIMEOUT):
            print(f"  WARNING: Tchamber did not reach {temp} °C within {TEMP_TIMEOUT} s; continuing")

        for rh in RH_VALUES:
            li.set(rh_air=rh)                   # was RHcham (a read-only key -> KeyError)
            print(f"\nRH -> {rh} %; waiting to stabilise ...")
            if not li.wait_stable("RHcham", rh, tol=RH_TOL, timeout_s=RH_TIMEOUT):
                print(f"  WARNING: RHcham did not reach {rh} % within {RH_TIMEOUT} s; continuing")

            for rep in range(REPEATS):
                for sp in CO2_S_VALUES:
                    try:
                        time.sleep(1)  # just a moment before changing the setpoint
                        li.set(co2_s=sp)
                        print(f"\nco2_s -> {sp} ppm; recording {MEASUREMENTS} x {DELAY_BETWEEN_MEASUREMENTS} s response ...")
                        for i in range(MEASUREMENTS):
                            rec = acquire(li, dot, co2_s_setpoint=sp, save=False)   # collect; no per-point file
                            rec["temp"] = temp
                            rec["rh"] = rh
                            rec["rep"] = rep
                            rec["measurement"] = i
                            experiment["records"].append(rec)
                            exp_path.write_text(json.dumps(experiment, indent=2), encoding="utf-8")  # all-in-one, after each point
                            print(f"  saved point {len(experiment['records'])}/{TOTAL} -> {exp_path.name}")
                            if i < MEASUREMENTS - 1:   # no trailing sleep after the last measurement
                                time.sleep(DELAY_BETWEEN_MEASUREMENTS)
                    except Exception as e:
                        print(f"  ERROR: {e}; skipping this point")
finally:
    # Always runs -- normal completion, Ctrl-C, or any escaping error -- so the
    # chamber is never left parked at the last setpoint and the JSON always gets
    # an end marker + final flush.
    try:
        li.set(co2_s=419)  # back to ambient
    except Exception as e:
        print(f"  WARNING: could not restore co2_s to ambient: {e}")
    experiment["experiment_end"] = datetime.now().isoformat(timespec="seconds")
    exp_path.write_text(json.dumps(experiment, indent=2), encoding="utf-8")
    print(f"\nDone -- {len(experiment['records'])} measurements in {exp_path}")


In [ ]:
# Full experiment: iterate co2_s setpoints. For each: set -> stabilise -> +1 min -> acquire.
# All points from this run go into ONE timestamped JSON file, rewritten after every
# point so partial data survives an interruption. Each record carries its own timestamp.
TEMP_VALUES = [17,23,31]
RH_VALUES = [40, 50, 60, 70]   # %RH; edit as needed
CO2_S_VALUES = [300,330,360,390,420,450,480,510,480,450,420,390,360,330]   # ppm; edit as needed
STABILISE_TOL = 1          # ppm window counted as "stable"
STABILISE_TIMEOUT = 300    # s max wait for stability
EXTRA_WAIT = 2*60            # s extra settle after stable
REPEATS = 4


outdir = Path("co2dot_data")
outdir.mkdir(parents=True, exist_ok=True)
started = datetime.now()
exp_path = outdir / (started.strftime("%Y-%m-%d_%H-%M-%S") + "_co2response.json")

experiment = {
    "experiment_start": started.isoformat(timespec="seconds"),
    "co2_s_values": CO2_S_VALUES,
    "stabilise_tol": STABILISE_TOL,
    "stabilise_timeout_s": STABILISE_TIMEOUT,
    "extra_wait_s": EXTRA_WAIT,
    "records": [],
}


rep = 0
try:
    for rep in range(REPEATS):
        for sp in CO2_S_VALUES:
            try:
                time.sleep(1)  # just a moment before changing the setpoint
                li.set(co2_s=sp)
                print(f"\nco2_s -> {sp} ppm; waiting to stabilise ...")
                ok = li.wait_stable("CO2_s", sp, tol=STABILISE_TOL, timeout_s=STABILISE_TIMEOUT)
                print(f"  {'stable' if ok else 'TIMED OUT (measuring anyway)'}; "
                    f"CO2_s={li.get('CO2_s')}; extra {EXTRA_WAIT} s ...")
                time.sleep(EXTRA_WAIT)
                rec = acquire(li, dot, co2_s_setpoint=sp, save=False)   # collect; no per-point file
                rec["stable"] = ok
                rec["rep"] = rep
                experiment["records"].append(rec)
                exp_path.write_text(json.dumps(experiment, indent=2), encoding="utf-8")  # all-in-one, after each point
                print(f"  saved point {len(experiment['records'])}/{len(CO2_S_VALUES) * 3} -> {exp_path.name}")
            except Exception as e:
                print(f"  ERROR: {e}; skipping this point")
finally:
    # Always runs -- normal completion, Ctrl-C, or any escaping error -- so the
    # chamber is never left parked at the last setpoint and the JSON always gets
    # an end marker + final flush.
    try:
        li.set(co2_s=419)  # back to ambient
    except Exception as e:
        print(f"  WARNING: could not restore co2_s to ambient: {e}")
    experiment["experiment_end"] = datetime.now().isoformat(timespec="seconds")
    exp_path.write_text(json.dumps(experiment, indent=2), encoding="utf-8")
    print(f"\nDone -- {len(experiment['records'])} measurements in {exp_path}")



co2_s -> 300 ppm; waiting to stabilise ...
  stable; CO2_s=300.296; extra 120 s ...
  saved point 1/42 -> 2026-06-18_11-59-45_co2response.json

co2_s -> 330 ppm; waiting to stabilise ...
  stable; CO2_s=329.163; extra 120 s ...
  saved point 2/42 -> 2026-06-18_11-59-45_co2response.json

co2_s -> 360 ppm; waiting to stabilise ...
  stable; CO2_s=359.232; extra 120 s ...
  saved point 3/42 -> 2026-06-18_11-59-45_co2response.json

co2_s -> 390 ppm; waiting to stabilise ...
  stable; CO2_s=389.342; extra 120 s ...
  saved point 4/42 -> 2026-06-18_11-59-45_co2response.json

co2_s -> 420 ppm; waiting to stabilise ...
  stable; CO2_s=419.353; extra 120 s ...
  saved point 5/42 -> 2026-06-18_11-59-45_co2response.json

co2_s -> 450 ppm; waiting to stabilise ...
  stable; CO2_s=449.04; extra 120 s ...
  saved point 6/42 -> 2026-06-18_11-59-45_co2response.json

co2_s -> 480 ppm; waiting to stabilise ...
  stable; CO2_s=479.102; extra 120 s ...
  saved point 7/42 -> 2026-06-18_11-59-45_co2respons

In [31]:
# Set chamber conditions. THIS DRIVES THE HARDWARE IMMEDIATELY.
# Supported keys: co2_r, co2_s, co2_pct, flow, flow_pct, tair, tleaf, txchg,
#                 rh_air, vpd_leaf, sd_air, h2o_r, h2o_s, fan_rpm.
# Pass a value of None to turn a controller off.
li.set(co2_s=380, tair=25, rh_air=50)

In [38]:
# Optionally block until a value settles, then read it back.
li.wait_stable("CO2_s", 380, tol=1, timeout_s=120)
print("CO2_s =", li.get("CO2_s"), "  Tleaf =", li.get("Tleaf"), "  A =", li.get("A"))

CO2_s = 380.934   Tleaf = 24.3604   A = -0.8313226594512364


In [ ]:
li.close()   # disconnect from the bus when done (or use `with LI6800.connect() as li:`)

### MQTT reference

**Setpoints** (`li.set(**kwargs)`, value = `None` turns the controller off):

| Key | Sets | Units |
|---|---|---|
| `co2_r` / `co2_s` | Reference / sample CO₂ | µmol mol⁻¹ |
| `co2_pct` | CO₂ mixer % | % |
| `tair` / `tleaf` / `txchg` | Air / leaf / exchanger temperature | °C |
| `rh_air` / `vpd_leaf` / `sd_air` | Humidity by RH / VPD / SD | % / kPa / kPa |
| `h2o_r` / `h2o_s` | Reference / sample H₂O | mmol mol⁻¹ |
| `flow` / `flow_pct` | Chamber flow | µmol s⁻¹ / % |
| `fan_rpm` | Fan speed | rpm |

Light/`Qin` is not in the typed setters yet (the LI-6800 routes light across head / console / fluorometer sources); use `li.set_raw(topic, payload)` or a background program for now.

**Reading:** `li.read()` returns a dict; `li.get("A")` one value; `li.raw(topic)` a subscribed topic's last payload. Common keys: `CO2_r`, `CO2_s`, `H2O_r`, `H2O_s`, `Tleaf`, `Tchamber`, `RHcham`, `VPDleaf`, `Flow`, `Fan_speed`, `Press`, and computed `A`, `gsw`, `gbw`, `Ci`, `E`.

**Architecture:** the broker (Mosquitto, port 1883) runs on the sensor **head** (`li6850`); the console (`li6860`) and fluorometer (`li6840`) are clients. Topic strings can drift between Bluestem firmware versions — re-verify with `mqtt_sniff.py` if a future update breaks something.

**Background programs (experimental):** `li.run_bp("/home/licor/apps/.../X.py")`, `li.run_steps(steps)`, `li.stop_bp()`.

## Fallback: SSH + `RemoteEnvMeasure` background program

The original path: JSON commands shipped over SSH to a resident background program. Use it only if the MQTT route is unavailable. It **requires `RemoteEnvMeasure.py` to be running** on the console (start it from Bluestem ▸ Background Programs); `li_ssh.bp_running()` checks that. First connect installs an SSH key (factory password: `licor`); later runs are passwordless.

In [ ]:
from li_connect import LiCor

li_ssh = LiCor.connect()          # enter password 'licor' on first run; passwordless after
print("background program running:", li_ssh.bp_running())

In [ ]:
# Read current values (no setpoint change, no log record):
print(li_ssh.read())

# Set conditions and log one record (requires an open log file on the console):
ack = li_ssh.measure(co2_r=399, tair=25, rh_air=50, wait_for_co2=True, co2_tol=5, wait_s=1, log=True)
ack